# Jailbreak Pipeline Studio - try in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rkstlohchab/jailbreak-pipeline-studio/blob/main/studio/notebooks/studio.ipynb)

Zero-install trial of the whole studio app. `Runtime > Run All`, then click the URL that prints at the bottom.

**Requirements**
- Free Colab account
- ~5 minutes for first boot (installs Node, pnpm, Python deps, builds the Next.js frontend)
- Colab GPU not required - this is a CPU-only app

**What boots**
- FastAPI backend (port 8000) with the full taxonomy + 10 pre-ingested sessions
- Next.js 16 / React 19 frontend (port 3000) - pipeline, techniques, failures, matrix, corpus, dataset views

The notebook builds the frontend against a Colab-proxied backend URL so client-side `fetch` works from your browser. No API key is required for the read-only views; live-run / classification features need an Anthropic key pasted into the Playground UI.


In [ ]:
# Env check: uname, Python, Node. Installs Node 20 LTS if missing or <20.
import subprocess, sys, shutil, platform, os

def sh(cmd, check=True):
    print(">>", " ".join(cmd) if isinstance(cmd, list) else cmd)
    r = subprocess.run(cmd, shell=isinstance(cmd, str), check=check,
                       capture_output=True, text=True)
    if r.stdout: print(r.stdout.rstrip())
    if r.stderr: print(r.stderr.rstrip())
    return r

print("uname:", platform.uname())
print("python:", sys.version.split()[0])

node_path = shutil.which("node")
need_node = True
if node_path:
    try:
        v = subprocess.check_output(["node", "--version"], text=True).strip()
        print("node:", v)
        major = int(v.lstrip("v").split(".")[0])
        if major >= 20:
            need_node = False
    except Exception as e:
        print("node version probe failed:", e)

if need_node:
    print("Installing Node.js 20 LTS via NodeSource...")
    sh("curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -")
    sh(["sudo", "apt-get", "install", "-y", "nodejs"])
    sh(["node", "--version"])
    sh(["npm", "--version"])
else:
    print("Node.js 20+ already present.")


In [ ]:
import os, subprocess, pathlib
REPO = "https://github.com/rkstlohchab/jailbreak-pipeline-studio.git"
ROOT = pathlib.Path("/content/jailbreak-pipeline-studio")
if not ROOT.exists():
    subprocess.check_call(["git", "clone", "--depth=1", REPO, str(ROOT)])
else:
    print("repo already present, pulling latest...")
    subprocess.check_call(["git", "-C", str(ROOT), "pull", "--ff-only"])
os.chdir(ROOT)
print("cwd:", os.getcwd())
print("studio tree:")
subprocess.check_call(["ls", "-la", "studio"])


In [ ]:
# Install pnpm@10 globally, then frontend deps. First run is slow (~1-3 min).
import os, subprocess, time, pathlib
ROOT = pathlib.Path("/content/jailbreak-pipeline-studio")

t0 = time.time()
subprocess.check_call(["npm", "install", "-g", "pnpm@10"])
print(f"pnpm installed in {time.time()-t0:.1f}s")

os.chdir(ROOT / "studio" / "frontend")
t0 = time.time()
subprocess.check_call(["pnpm", "install", "--frozen-lockfile"])
print(f"pnpm install completed in {time.time()-t0:.1f}s")


In [ ]:
import os, subprocess, pathlib
ROOT = pathlib.Path("/content/jailbreak-pipeline-studio")
os.chdir(ROOT / "studio" / "backend")
subprocess.check_call([
    "pip", "install", "--quiet",
    "fastapi>=0.115",
    "uvicorn[standard]>=0.30",
    "pydantic>=2.7",
    "python-dotenv>=1.0.0",
    "requests>=2.31.0",
    "websockets>=12.0",
    "sse-starlette>=2.1",
])
print("backend deps installed.")


In [ ]:
# Build the frontend with NEXT_PUBLIC_API_BASE pre-baked.
#
# The client bundle hardcodes the backend URL at build time. In Colab the user's
# browser cannot reach 127.0.0.1:8000 on the kernel VM, so we point the client
# at Colab's port-8000 proxy URL instead. If this isn't Colab, we fall back
# to http://127.0.0.1:8000 (local dev).
import os, subprocess, time, pathlib
ROOT = pathlib.Path("/content/jailbreak-pipeline-studio")

api_base = None
try:
    from google.colab import output as colab_output  # type: ignore
    api_base = colab_output.eval_js(
        'google.colab.kernel.proxyPort(8000, {"cache": true})'
    )
    print("Colab proxy URL for backend (port 8000):", api_base)
except Exception as e:
    print("Not running on Colab or proxy unavailable; using 127.0.0.1:8000. Reason:", e)

env = os.environ.copy()
if api_base:
    # Strip trailing slash so `${API_BASE}/api/...` concatenates cleanly.
    env["NEXT_PUBLIC_API_BASE"] = api_base.rstrip("/")
# Cap Node heap so pnpm build doesn't OOM on low-RAM Colab boxes.
env.setdefault("NODE_OPTIONS", "--max-old-space-size=3072")

os.chdir(ROOT / "studio" / "frontend")
t0 = time.time()
subprocess.check_call(["pnpm", "build"], env=env)
print(f"pnpm build completed in {time.time()-t0:.1f}s")


In [ ]:
# Start FastAPI backend in the background, then health-check it.
import os, subprocess, time, pathlib, urllib.request, json
ROOT = pathlib.Path("/content/jailbreak-pipeline-studio")
os.chdir(ROOT / "studio" / "backend")

backend_log = open("/tmp/backend.log", "w")
backend_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app",
     "--host", "127.0.0.1", "--port", "8000"],
    stdout=backend_log, stderr=subprocess.STDOUT,
)
print("backend pid:", backend_proc.pid)

# Give uvicorn a moment to bind.
healthy = False
for attempt in range(15):
    time.sleep(1)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2) as r:
            print("backend health:", json.load(r))
            healthy = True
            break
    except Exception as e:
        if attempt == 14:
            print("backend not healthy after 15s:", e)

if not healthy:
    print("---- /tmp/backend.log tail ----")
    backend_log.flush()
    try:
        print(open("/tmp/backend.log").read()[-2000:])
    except Exception as e:
        print("could not read log:", e)


In [ ]:
# Start Next.js (production) on port 3000 in the background.
import os, subprocess, time, pathlib, urllib.request
ROOT = pathlib.Path("/content/jailbreak-pipeline-studio")
os.chdir(ROOT / "studio" / "frontend")

frontend_log = open("/tmp/frontend.log", "w")
# `pnpm start` runs `next start` which serves the production build on 0.0.0.0:3000.
env = os.environ.copy()
env.setdefault("PORT", "3000")
env.setdefault("HOSTNAME", "0.0.0.0")
frontend_proc = subprocess.Popen(
    ["pnpm", "start"],
    stdout=frontend_log, stderr=subprocess.STDOUT, env=env,
)
print("frontend pid:", frontend_proc.pid)

ready = False
for attempt in range(45):
    time.sleep(1)
    try:
        with urllib.request.urlopen("http://127.0.0.1:3000/", timeout=2) as r:
            if r.status < 500:
                print(f"frontend responded with HTTP {r.status} after {attempt+1}s")
                ready = True
                break
    except Exception as e:
        if attempt == 44:
            print("frontend not ready after 45s:", e)

if not ready:
    print("---- /tmp/frontend.log tail ----")
    frontend_log.flush()
    try:
        print(open("/tmp/frontend.log").read()[-2000:])
    except Exception as e:
        print("could not read log:", e)


In [ ]:
# Primary: Colab's own kernel-port proxy. Button opens the UI in a new tab;
# the iframe below shows it inline. If both fail, scroll to the cloudflared
# fallback block at the bottom of this cell.
try:
    from google.colab import output as colab_output  # type: ignore
    from IPython.display import HTML, display

    colab_output.serve_kernel_port_as_window(3000, anchor_text="OPEN STUDIO IN NEW TAB")

    frontend_url = colab_output.eval_js(
        'google.colab.kernel.proxyPort(3000, {"cache": true})'
    )
    print("public frontend URL:", frontend_url)

    display(HTML(
        '<p style="font-family:monospace;font-size:13px;margin:10px 0">'
        'If the button above does not work, the iframe below is the studio UI. '
        'First load may take ~10s as Next.js warms the first route.'
        '</p>'
    ))
    colab_output.serve_kernel_port_as_iframe(3000, height=900)
except Exception as e:
    print("Colab port-forward helpers failed:", e)
    print("Falling back to cloudflared quick tunnel...")
    import subprocess, os, time, re, pathlib
    CFD = "/tmp/cloudflared"
    if not pathlib.Path(CFD).exists():
        subprocess.check_call([
            "wget", "-q", "-O", CFD,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ])
        subprocess.check_call(["chmod", "+x", CFD])
    cfd_log = open("/tmp/cloudflared.log", "w")
    cfd_proc = subprocess.Popen(
        [CFD, "tunnel", "--url", "http://localhost:3000", "--no-autoupdate"],
        stdout=cfd_log, stderr=subprocess.STDOUT,
    )
    print("cloudflared pid:", cfd_proc.pid)
    # Scrape the trycloudflare.com URL from the log.
    url = None
    for _ in range(30):
        time.sleep(1)
        cfd_log.flush()
        try:
            txt = open("/tmp/cloudflared.log").read()
            m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", txt)
            if m:
                url = m.group(0)
                break
        except Exception:
            pass
    if url:
        print("public URL via cloudflared:", url)
    else:
        print("cloudflared did not surface a URL in 30s. Tail of its log:")
        print(open("/tmp/cloudflared.log").read()[-2000:])


## What's loaded

- `/` - landing page with hero + 7 view tiles
- `/pipeline` - 3D refusal-gate visualisation
- `/techniques` - 50-technique catalog
- `/failures` - 20 named failure cases
- `/matrix` - 17 model families x ask classes
- `/corpus` - 121 pre-classified jailbreak prompts
- `/dataset` - dataset exporter (OpenAI SFT / Anthropic / Alpaca / lessons / curriculum)

## Limitations in Colab

- Live LLM runs from the Playground require you to paste your Anthropic API key into the Playground UI (keys are never stored server-side, held in memory only).
- Classifying your own prompt corpus via `scripts/classify_corpus.py` needs a key too.
- Session ends when Colab runtime disconnects (~90 min idle); re-run all cells to restart.
- Colab's URL is temporary - do not share it publicly, it exposes your runtime.


## Troubleshooting

**`backend not healthy`** - inspect the tail of `/tmp/backend.log`. Most common causes: missing Python dep (re-run cell 5), port 8000 already bound by a previous run (restart the runtime), or an import error in `studio/backend/app/`.

**`pnpm install` hangs or fails** - transient network. Re-run cell 4. If the lockfile is rejected, the pinned pnpm version is out of sync; run `pnpm install` without `--frozen-lockfile`.

**Frontend `pnpm build` fails on OOM** - Colab's free tier sometimes allocates <4GB. Cell 6 already sets `NODE_OPTIONS=--max-old-space-size=3072`; drop it further to `2048` if needed, or switch runtime type to "High-RAM" if you have Colab Pro.

**Port 3000 iframe is blank or shows `fetch` errors in DevTools** - this means either (a) the backend is down (check cell 7), or (b) the frontend was built before the Colab proxy URL was resolved. Re-run cell 6 then cell 8. The built-in iframe can also be flaky; use the `OPEN STUDIO IN NEW TAB` button from cell 9 instead.

**`google.colab` module not found** - you're running this notebook outside Colab. Cell 6 and cell 9 fall back to `127.0.0.1` and `cloudflared` respectively; for a pure-local trial, follow `studio/README.md` instead.

**App loads but all API calls 404** - the frontend bundle is pointed at the wrong backend URL. Confirm `NEXT_PUBLIC_API_BASE` was set in cell 6 before `pnpm build` ran; re-run cells 6, 7, 8, 9 in order.
